In [ ]:
import sys as _sys
from pathlib import Path as _Path

_d = _Path.cwd().resolve()
while _d != _d.parent and not (_d / "Benchmarking" / "_paths.py").exists():
    _d = _d.parent
_sys.path.insert(0, str(_d / "Benchmarking"))
import _paths as _cg  # noqa: E402

In [ ]:
import gc
import os
import sys

import cell2location
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import scipy.stats as stats
import scvi
import torch
from anndata import AnnData
from fast_matrix_market import mmread
from matplotlib import rcParams
from scipy.spatial.distance import jensenshannon
from scipy.stats import pearsonr

# Set matplotlib parameters
rcParams["pdf.fonttype"] = 42  # Enables correct plotting of text for PDFs

In [ ]:
def load_wu_data(path, test_mode=False):
    matrix_path = os.path.join(path, "matrix.mtx")
    genes_path = os.path.join(path, "genes.tsv")
    barcodes_path = os.path.join(path, "barcodes.tsv")
    metadata_path = os.path.join(path, "metadata.csv")
    try:
        genes_df = pd.read_csv(genes_path, header=None, sep="\t")
        barcodes_df = pd.read_csv(barcodes_path, header=None)
        barcodes = barcodes_df[0]

        matrix = mmread(matrix_path).T.tocsc().tocsr().astype("float32")
        var_df = pd.DataFrame(index=genes_df[0])

        # Create the initial AnnData object
        adata = AnnData(matrix, obs=pd.DataFrame(index=barcodes), var=var_df)

        # Load the metadata
        metadata_df = pd.read_csv(metadata_path, index_col=0)
        adata.obs["patient_ID"] = metadata_df["orig.ident"]
        adata.obs["celltype"] = metadata_df["celltype_major"]
        adata.obs["tissue"] = "Cancer"
        adata.obs["subtype"] = metadata_df["subtype"]
        adata.obs.index.name = None
        if test_mode:
            # Downsample in test mode
            adata = adata[np.random.choice(adata.shape[0], 10000, replace=False)]

        return adata

    except Exception as e:
        print("Error in load_wu_data function:", e)
        raise


wu_path = str(_cg.root("DATA") / "BC-Datasets" / "Wu_2021" / "Wu_etal_2021_BRCA_scRNASeq")
test_mode = False

# load the ER+ reference dataset
wu_adata = load_wu_data(wu_path, test_mode)
wu_adata

In [ ]:
# subset Wu object to ER+ cells
wu_adata = wu_adata[wu_adata.obs["subtype"] == "ER+"]
wu_adata.var.index.name = "Index"
from cell2location.utils.filtering import filter_genes

adata_ref = wu_adata.copy()

adata_ref.var

In [ ]:
[gene.startswith("MT-") for gene in adata_ref.var_names]

In [ ]:
# find mitochondria-encoded (MT) genes
adata_ref.var["MT_gene"] = [gene.startswith("MT-") for gene in adata_ref.var_names]

# remove MT genes for spatial mapping (keeping their counts in the object)
adata_ref.obsm["MT"] = adata_ref[:, adata_ref.var["MT_gene"].values].X.toarray()
adata_ref = adata_ref[:, ~adata_ref.var["MT_gene"].values]

selected = filter_genes(adata_ref, cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12)

# filter the object
adata_ref = adata_ref[:, selected].copy()
adata_ref

In [ ]:
results_folder = str(_cg.root("DATA") / "Wu_Visium" / "Simulations" / "larger_ref")

# create paths and names to results folders for reference regression and cell2location models
ref_run_name = f"{results_folder}/reference_signatures"
run_name = f"{results_folder}/cell2location_map"

In [ ]:
# prepare anndata for the regression model
TF_CPP_MIN_LOG_LEVEL = 0
cell2location.models.RegressionModel.setup_anndata(
    adata=adata_ref,
    # 10X reaction / sample / batch
    batch_key="patient_ID",
    # cell type, covariate used for constructing signatures
    labels_key="celltype",
    # multiplicative technical effects (platform, 3' vs 5', donor effect)
    # categorical_covariate_keys=['patient_ID']
)

In [ ]:
# create the regression model
from cell2location.models import RegressionModel

mod = RegressionModel(adata_ref)

# view anndata_setup as a sanity check
mod.view_anndata_setup()

In [ ]:
mod.train(max_epochs=250, use_gpu=True)

In [ ]:
mod.plot_history()

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_ref = mod.export_posterior(adata_ref, sample_kwargs={"num_samples": 1000, "batch_size": 2500, "use_gpu": True})
# Save model
mod.save(f"{ref_run_name}", overwrite=True)

# Save anndata object with results
adata_file = f"{ref_run_name}/reference_major.h5ad"
adata_ref.write(adata_file)
adata_file

In [ ]:
mod.plot_QC()

In [ ]:
# export estimated expression in each cluster
if "means_per_cluster_mu_fg" in adata_ref.varm.keys():
    inf_aver = adata_ref.varm["means_per_cluster_mu_fg"][
        [f"means_per_cluster_mu_fg_{i}" for i in adata_ref.uns["mod"]["factor_names"]]
    ].copy()
else:
    inf_aver = adata_ref.var[[f"means_per_cluster_mu_fg_{i}" for i in adata_ref.uns["mod"]["factor_names"]]].copy()
inf_aver.columns = adata_ref.uns["mod"]["factor_names"]
inf_aver.iloc[0:5, 0:5]